# gzip - Rust

All 10 Rust examples from [docs/gzip.md](https://platob.github.io/yggdryl/gzip/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::gzip;

let encoded = gzip::dump(b"symbol,price\nAAPL,1\n")?;
assert_eq!(gzip::load(&encoded)?, b"symbol,price\nAAPL,1\n");

## Levels

In [ ]:
use yggdryl::gzip;
use yggdryl::Level;

let payload = b"AAPL,1\nAAPL,2\nAAPL,3\nAAPL,4\nAAPL,5\nAAPL,6\nAAPL,7\nAAPL,8\n";

let stored = gzip::dump_with_level(payload, Level::NONE)?;
let smallest = gzip::dump_with_level(payload, Level::BEST)?;
assert!(smallest.len() < stored.len());
assert_eq!(gzip::load(&stored)?, payload);
assert_eq!(gzip::load(&smallest)?, payload);

// One 0-to-9 scale, clamped at both ends, with 6 when nobody chooses.
assert_eq!(Level::DEFAULT.get(), 6);
assert_eq!(Level::new(12), Level::BEST);
assert_eq!(gzip::dump(payload)?, gzip::dump_with_level(payload, Level::DEFAULT)?);

## Streams

In [ ]:
use std::io::{Read, Write};
use yggdryl::gzip;

let mut target = Vec::new();
let mut encoder = gzip::writer(&mut target);
encoder.write_all(b"symbol,price\nAAPL,1\n")?;
encoder.finish()?;

let mut decoded = Vec::new();
gzip::reader(target.as_slice()).read_to_end(&mut decoded)?;
assert_eq!(decoded, b"symbol,price\nAAPL,1\n");

In [ ]:
use std::io::{Read, Write};
use yggdryl::{gzip, Level};

let mut target = Vec::new();
let mut encoder = gzip::writer_with_level(&mut target, Level::BEST);
encoder.write_all(b"symbol,price\nAAPL,1\n")?;
encoder.finish()?;

let mut head = [0_u8; 6];
gzip::reader(target.as_slice()).read_exact(&mut head)?;
assert_eq!(&head, b"symbol");

## A handle that hides the coding

In [ ]:
use yggdryl::gzip::{self, Gzip};
use yggdryl::io::{Buffer, IOBase};

let mut handle = Gzip::new(Buffer::new());
handle.write_all_bytes(b"symbol,price\nAAPL,1\n")?;
handle.flush()?;

// The wrapper reads and measures the plain bytes.
assert_eq!(handle.read_all_bytes()?, b"symbol,price\nAAPL,1\n");
assert_eq!(handle.size(), 20);

// The wrapped handle holds the gzip member.
let inner = handle.into_handle()?;
assert_eq!(gzip::load(inner.as_slice())?, b"symbol,price\nAAPL,1\n");

In [ ]:
use yggdryl::gzip::Gzip;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::Level;

let mut handle = Gzip::new(Buffer::new()).with_level(Level::BEST);
assert_eq!(handle.level(), Level::BEST);

handle.write_all_bytes(b"symbol,price\nAAPL,1\n")?;
handle.flush()?;
assert_eq!(handle.read_all_bytes()?, b"symbol,price\nAAPL,1\n");

## A `.gz` name is enough

In [ ]:
use yggdryl::generic::Coded;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{MediaType, MimeType};

// `.gz` is the last suffix, so gzip is the outermost coding of a CSV.
let named = MediaType::from_file_name("trades.csv.gz");
assert_eq!(named.base(), &MimeType::CSV);
assert_eq!(yggdryl::Codec::from_media_type(&named), yggdryl::Codec::Gzip);

// A handle that declares that media type picks its own coding.
let mut handle = Coded::infer(Buffer::new().with_media_type(named));
assert_eq!(handle.codec(), yggdryl::Codec::Gzip);

handle.write_all_bytes(b"symbol,price\nAAPL,1\n")?;
handle.flush()?;
assert_eq!(yggdryl::gzip::load(handle.handle().as_slice())?, b"symbol,price\nAAPL,1\n");

In [ ]:
use yggdryl::gzip::Gzip;
use yggdryl::io::{Buffer, IOBase};
use yggdryl::{MediaType, MimeType};

let buffer = Buffer::new().with_media_type(MediaType::from_file_name("trades.csv.gz"));
assert!(buffer.media_type().is_encoded());

let handle = Gzip::new(buffer);
assert_eq!(handle.media_type().base(), &MimeType::CSV);
assert!(!handle.media_type().is_encoded());

## Failures

In [ ]:
use yggdryl::gzip;

assert!(gzip::load(b"definitely not a compressed payload").is_err());

In [ ]:
use yggdryl::gzip::Gzip;
use yggdryl::io::{Buffer, IOBase};

let handle = Gzip::new(Buffer::new());
assert_eq!(handle.size(), 0);
assert!(handle.read_all_bytes()?.is_empty());